# 3D Body Reconstruction for Soccer Players using **SAM 3D Body**

This notebook is part of a "Computer Vision" prototype for testing out 3D body reconstruction of soccer players both in-game and during training by leveraging SotA SAM 3D Body.

### Requirements
- python 3.12 (3.11 should be fine, although untested)
- torch with cuda 12.6
- rest is installed here

### Setup

In [1]:
!pip install --quiet timm einops pycocotools huggingface_hub opencv-python triton psutil notebook boxmot braceexpand matplotlib roma omegaconf pytorch_lightning yacs pyrender fvcore black iopath==0.1.7 cloudpickle hydra-core tensorboard setuptool<82

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 29.6 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 110.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 69.2 MB/s eta 0:00:0

In [2]:
!pip install --quiet 'git+https://github.com/facebookresearch/detectron2.git@a1ce2f9' --no-build-isolation --no-deps

  Preparing metadata (setup.py) ... done


In [3]:
!pip install --quiet git+https://github.com/microsoft/MoGe.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 5.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 3.1 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade gradio

In [ ]:
from huggingface_hub import login
login(token="HF_TOKEN")

### Imports

In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from notebook.utils import setup_sam_3d_body
from tools.vis_utils import visualize_sample_together, visualize_sample
from sam_3d_body.visualization.renderer import Renderer
from sam_3d_body.visualization.skeleton_visualizer import SkeletonVisualizer
from sam_3d_body.metadata.mhr70 import pose_info as mhr70_pose_info
from boxmot import ByteTrack

### Load visualizer & estimator

In [4]:
LIGHT_BLUE = (0.65098039, 0.74117647, 0.85882353)

skeleton_visualizer = SkeletonVisualizer(line_width=2, radius=5)
skeleton_visualizer.set_pose_meta(mhr70_pose_info)

In [29]:
# Set up the estimator
estimator = setup_sam_3d_body(hf_repo_id="facebook/sam-3d-body-dinov3")

Loading SAM 3D Body model from facebook/sam-3d-body-dinov3...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading SAM 3D Body model...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov3_main
Ignored kwargs: {'drop_path': 0.1}
The model and loaded state dict do not match exactly

missing keys in source state_dict: backbone.encoder.mask_token, head_pose.hand_pose_comps_ori, head_pose.mhr.face_expressions_model.shape_vectors, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.0.sparse_indices, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.0.sparse_weight, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.2.weight, head_pose.mhr.character_torch.skeleton.joint_translation_offsets, head_pose.mhr.character_torch.skeleton.joint_prerotations, head_pose.mhr.character_torch.skeleton.pmi, head_pose.mhr.character_torch.skeleton.joint_parents, head_pose.mhr.character_torch.mesh.rest_vertices, head_pose.mhr.character_torch.mesh.faces, head_pose.mhr.character_torch.mesh.texcoords, head_pose.mhr.character_torch.mesh.texcoord_faces, head_pose.mhr.character_torch.parameter_transform.parameter_tra

Loading human detector from vitdet...
########### Using human detector: ViTDet...


/venv/main/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
model_final_f05665.pkl: 2.77GB [00:45, 60.8MB/s]                               


Loading FOV estimator from moge2...
########### Using fov estimator: MoGe2...


model.pt:   0%|          | 0.00/1.32G [00:00<?, ?B/s]

Mask-condition inference is not supported...
Setup complete!
  Human detector: ✓
  Human segmentor: ✗ (mask inference disabled)
  FOV estimator: ✓


### Functions

In [6]:
def process_image(img_path:str):
    """
    Simple processing of an image.
    
    Args:
    img_path:

    Returns:
    rendered image
    """
    img_bgr = cv2.imread(img_path)
    outputs = estimator.process_one_image(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    
    # Visualize and save results
    rend_img = visualize_sample_together(img_bgr, outputs, estimator.faces)
    cv2.imwrite("output.jpg", rend_img.astype(np.uint8))
    
    return rend_img

In [7]:
def save_obj(outputs, faces, output_path="output.obj"):
    """
    create a 3D object file of the player's mesh

    Args:
    outputs
    faces
    output_path

    Returns:
    3D object file
    """
    all_vertices = []
    for pid, person_output in enumerate(outputs):
        verts = person_output["pred_vertices"] + person_output["pred_cam_t"]
        all_vertices.append(verts)
    all_vertices = np.concatenate(all_vertices, axis=0)
    all_vertices[:, 1] *= -1  # flip Y
    all_vertices[:, 2] *= -1  # flip Z

    with open(output_path, "w") as f:
        for v in all_vertices:
            f.write(f"v {v[0]} {v[1]} {v[2]}\n")
        # .obj faces are 1-indexed
        for pid in range(len(outputs)):
            offset = len(person_output["pred_vertices"]) * pid
            for face in faces:
                f1, f2, f3 = face + offset + 1
                f.write(f"f {f1} {f2} {f3}\n")

    return output_path

In [8]:
def process_video(video_path: str, output_path: str = "output_video.mp4"):
    """
    Read a video frame by frame, run the SAM 3D body estimator on each frame,
    and write the visualised results to a new video file.

    Args:
        video_path:  Path to the input video file.
        output_path: Path for the output video (default: 'output_video.mp4').
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"Input  : {video_path}")
    print(f"Frames : {total}  |  FPS : {fps:.2f}  |  Size : {width}x{height}")

    # We don't know the output frame size until we process the first frame,
    # so defer writer creation until after the first successful render.
    writer = None
    out_w, out_h = None, None

    try:
        for frame_idx in range(total if total > 0 else int(1e9)):
            ret, frame_bgr = cap.read()
            if not ret:
                break

            # --- run estimator ---
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            outputs   = estimator.process_one_image(frame_rgb)

            # --- render overlay ---
            rend = visualize_sample_together(frame_bgr, outputs, estimator.faces)
            rend_bgr = rend.astype(np.uint8)   # already BGR from visualize_sample_together

            # --- init writer on first frame ---
            if writer is None:
                out_h, out_w = rend_bgr.shape[:2]
                fourcc = cv2.VideoWriter_fourcc(*"mp4v")
                writer = cv2.VideoWriter(output_path, fourcc, fps, (out_w, out_h))
                print(f"Output : {output_path}  |  Size : {out_w}x{out_h}")

            writer.write(rend_bgr)

            if (frame_idx + 1) % 10 == 0:
                print(f"  processed {frame_idx + 1}/{total} frames…")

    finally:
        cap.release()
        if writer is not None:
            writer.release()

    print(f"Done – saved to {output_path}")
    return output_path

In [9]:
# ── MHR70 joint indices (Multi-HMR / SMPL-X convention) ─────────────────────
# These are the indices into pred_keypoints_2d / pred_keypoints_3d
J = {
    "pelvis":       0,
    "l_hip":        1,   "r_hip":        2,
    "spine1":       3,
    "l_knee":       4,   "r_knee":        5,
    "spine2":       6,
    "l_ankle":      7,   "r_ankle":       8,
    "spine3":       9,
    "l_foot":      10,   "r_foot":       11,
    "neck":        12,
    "l_collar":    13,   "r_collar":     14,
    "head":        15,
    "l_shoulder":  16,   "r_shoulder":   17,
    "l_elbow":     18,   "r_elbow":      19,
    "l_wrist":     20,   "r_wrist":      21,
}

# ── Geometry helpers ─────────────────────────────────────────────────────────

def angle_between(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    """Angle at joint b formed by segments b→a and b→c, in degrees."""
    v1 = a - b;  v2 = c - b
    cos_a = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1, 1))))


def valgus_angle(hip: np.ndarray, knee: np.ndarray, ankle: np.ndarray) -> float:
    """
    Knee valgus: medial deviation of the knee from the hip-ankle line.
    Computed in the frontal plane (XY). Positive = valgus (knee caves in).
    """
    ref = ankle - hip
    dev = knee  - hip
    # Project onto frontal plane (ignore Z / depth)
    ref2 = ref[:2];  dev2 = dev[:2]
    if np.linalg.norm(ref2) < 1e-6 or np.linalg.norm(dev2) < 1e-6:
        return 0.0
    cos_a = np.dot(ref2, dev2) / (np.linalg.norm(ref2) * np.linalg.norm(dev2) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1, 1))))


# ── Per-person biomechanics ──────────────────────────────────────────────────

def compute_biomechanics(person_output: dict) -> dict:
    """
    Compute biomechanical metrics from a single person's 3D keypoints.
    Uses pred_keypoints_3d (camera-space 3D) when available,
    falls back to pred_keypoints_2d.

    Returns a dict with:
        left_knee_deg, right_knee_deg   – knee flexion angles
        l_valgus, r_valgus              – knee valgus angles (deg)
        knee_width_ratio                – knee-width / hip-width
        hip_drop_cm                     – pelvic drop (L vs R hip height, cm)
        trunk_lean_deg                  – trunk lean from vertical
        asym                            – |left_knee - right_knee|
        low_confidence                  – True if keypoints look unreliable
    """
    # Prefer 3D keypoints; fall back to 2D
    kp = person_output.get("pred_keypoints_3d", person_output.get("pred_keypoints_2d"))
    use_3d = "pred_keypoints_3d" in person_output
    scale = 100.0  # arbitrary → cm-like units for hip_drop when using 3D

    def pt(name):
        return kp[J[name]]

    # ── Knee flexion (hip-knee-ankle angle) ──────────────────────────────────
    l_knee_deg = angle_between(pt("l_hip"),  pt("l_knee"),  pt("l_ankle"))
    r_knee_deg = angle_between(pt("r_hip"),  pt("r_knee"),  pt("r_ankle"))

    # ── Valgus ───────────────────────────────────────────────────────────────
    l_val = valgus_angle(pt("l_hip"), pt("l_knee"), pt("l_ankle"))
    r_val = valgus_angle(pt("r_hip"), pt("r_knee"), pt("r_ankle"))

    # ── Knee-width ratio ─────────────────────────────────────────────────────
    knee_w = np.linalg.norm(pt("l_knee")[:2] - pt("r_knee")[:2])
    hip_w  = np.linalg.norm(pt("l_hip")[:2]  - pt("r_hip")[:2])
    kwr    = float(knee_w / (hip_w + 1e-8))

    # ── Hip drop (pelvic tilt in frontal plane) ───────────────────────────────
    # Difference in Y (vertical) between left and right hip — positive = L drops
    hip_drop = float((pt("l_hip")[1] - pt("r_hip")[1]) * scale)

    # ── Trunk lean from vertical ─────────────────────────────────────────────
    # Vector from pelvis to neck; angle with the vertical axis
    trunk_vec = pt("neck") - pt("pelvis")
    vertical  = np.array([0.0, 1.0, 0.0]) if use_3d else np.array([0.0, 1.0])
    trunk_vec_n = trunk_vec[:len(vertical)]
    cos_a = np.dot(trunk_vec_n, vertical) / (np.linalg.norm(trunk_vec_n) + 1e-8)
    trunk_lean = float(np.degrees(np.arccos(np.clip(cos_a, -1, 1))))

    # ── Confidence gate ───────────────────────────────────────────────────────
    # Flag frames where both knees are nearly fully extended (likely occluded / bad fit)
    low_conf = (l_knee_deg > 170 and r_knee_deg > 170)

    return {
        "left_knee_deg":  l_knee_deg,
        "right_knee_deg": r_knee_deg,
        "l_valgus":       l_val,
        "r_valgus":       r_val,
        "knee_width_ratio": kwr,
        "hip_drop_cm":    hip_drop,
        "trunk_lean_deg": trunk_lean,
        "asym":           abs(l_knee_deg - r_knee_deg),
        "low_confidence": low_conf,
    }


# ── Overlay renderer ─────────────────────────────────────────────────────────

def draw_metrics_overlay(frame: np.ndarray, metrics: dict, pos: tuple = (20, 40)) -> np.ndarray:
    """
    Draw biomechanical metrics as a HUD overlay on a frame (in-place).
    Colour-codes values: green = OK, orange = mild, red = concerning.
    """
    out   = frame.copy()
    x, y  = pos
    lh    = 30   # line height px
    fs    = 0.65 # font scale
    fw    = 1    # font weight
    font  = cv2.FONT_HERSHEY_SIMPLEX

    def colour(val, warn, bad):
        if abs(val) >= bad:   return (0, 0, 220)    # red
        if abs(val) >= warn:  return (0, 140, 255)   # orange
        return (0, 210, 0)                           # green

    if metrics.get("low_confidence"):
        cv2.putText(out, "LOW CONFIDENCE", (x, y), font, fs, (0,0,220), fw, cv2.LINE_AA)
        return out

    lines = [
        ("L-Knee",   f"{metrics['left_knee_deg']:.1f}deg",  colour(180-metrics['left_knee_deg'],  20, 40)),
        ("R-Knee",   f"{metrics['right_knee_deg']:.1f}deg", colour(180-metrics['right_knee_deg'], 20, 40)),
        ("L-Val",    f"{metrics['l_valgus']:.1f}deg",       colour(metrics['l_valgus'],  10, 20)),
        ("R-Val",    f"{metrics['r_valgus']:.1f}deg",       colour(metrics['r_valgus'],  10, 20)),
        ("KWR",      f"{metrics['knee_width_ratio']:.2f}",  colour(abs(metrics['knee_width_ratio']-1.0), 0.3, 0.6)),
        ("HipDrop",  f"{metrics['hip_drop_cm']:.1f}cm",     colour(abs(metrics['hip_drop_cm']),  3, 6)),
        ("Trunk",    f"{metrics['trunk_lean_deg']:.1f}deg", colour(metrics['trunk_lean_deg'],    20, 40)),
        ("Asym",     f"{metrics['asym']:.1f}deg",           colour(metrics['asym'],              15, 30)),
    ]

    for label, value, col in lines:
        text = f"{label}: {value}"
        cv2.putText(out, text, (x, y), font, fs, col, fw, cv2.LINE_AA)
        y += lh

    return out

In [10]:
def render_mesh_only(outputs, faces, img_h: int, img_w: int) -> np.ndarray:
    """
    Render the reconstructed 3D body mesh(es) onto a black canvas,
    then draw 2D joint skeletons and biomechanical metrics on top.
    All coordinates must be in the same space as img_h/img_w.
    """
    all_depths = np.stack([p["pred_cam_t"] for p in outputs], axis=0)[:, 2]
    outputs_sorted = [outputs[idx] for idx in np.argsort(-all_depths)]

    all_pred_vertices, all_faces = [], []
    for pid, person_output in enumerate(outputs_sorted):
        all_pred_vertices.append(person_output["pred_vertices"] + person_output["pred_cam_t"])
        all_faces.append(faces + len(person_output["pred_vertices"]) * pid)
    all_pred_vertices = np.concatenate(all_pred_vertices, axis=0)
    all_faces         = np.concatenate(all_faces, axis=0)

    fake_pred_cam_t = (
        np.max(all_pred_vertices[-2 * 18439:], axis=0)
        + np.min(all_pred_vertices[-2 * 18439:], axis=0)
    ) / 2
    all_pred_vertices = all_pred_vertices - fake_pred_cam_t

    black_img = np.zeros((img_h, img_w, 3), dtype=np.uint8)
    renderer  = Renderer(focal_length=outputs_sorted[-1]["focal_length"], faces=all_faces)
    rend = (
        renderer(
            all_pred_vertices, fake_pred_cam_t, black_img,
            mesh_base_color=LIGHT_BLUE, scene_bg_color=(0, 0, 0),
        ) * 255
    ).astype(np.uint8)

    # Skeleton + metrics — keypoints must already be in img_h/img_w space
    for person_output in outputs_sorted:
        kp2d = person_output["pred_keypoints_2d"]
        kp2d = np.concatenate([kp2d, np.ones((kp2d.shape[0], 1))], axis=-1)
        rend = skeleton_visualizer.draw_skeleton(rend, kp2d)
        metrics = compute_biomechanics(person_output)
        rend    = draw_metrics_overlay(rend, metrics)

    return rend


def process_video_mesh(video_path: str, output_path: str = "output_video.mp4"):
    """
    Read a video frame by frame, run the SAM 3D body estimator on each frame,
    render the reconstructed mesh with joint skeleton and biomechanical metrics
    overlay onto a black background, and write the result to a new video file.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"Input  : {video_path}")
    print(f"Frames : {total}  |  FPS : {fps:.2f}  |  Size : {width}x{height}")

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    print(f"Output : {output_path}")

    try:
        for frame_idx in range(total if total > 0 else int(1e9)):
            ret, frame_bgr = cap.read()
            if not ret:
                break

            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            outputs   = estimator.process_one_image(frame_rgb)

            if outputs:
                rend_bgr = render_mesh_only(outputs, estimator.faces, height, width)
            else:
                rend_bgr = np.zeros((height, width, 3), dtype=np.uint8)

            writer.write(rend_bgr)

            if (frame_idx + 1) % 10 == 0:
                print(f"  processed {frame_idx + 1}/{total} frames…")

    finally:
        cap.release()
        writer.release()

    print(f"Done – saved to {output_path}")
    return output_path

In [11]:
# ── Colour-based jersey fingerprint ─────────────────────────────────────────

def get_jersey_color(frame_bgr: np.ndarray, bbox: np.ndarray) -> np.ndarray:
    """
    Return the mean HSV colour of the torso region of a bounding box.
    We sample the middle third vertically (avoids head and legs).
    """
    x1, y1, x2, y2 = bbox[:4].astype(int)
    h = y2 - y1
    torso = frame_bgr[y1 + h // 3 : y1 + 2 * h // 3, x1:x2]
    if torso.size == 0:
        return np.zeros(3)
    hsv = cv2.cvtColor(torso, cv2.COLOR_BGR2HSV)
    return hsv.mean(axis=(0, 1))   # (H, S, V)


def color_distance(c1: np.ndarray, c2: np.ndarray) -> float:
    """Weighted HSV distance — hue counts more than value."""
    weights = np.array([2.0, 1.0, 0.5])
    return float(np.linalg.norm((c1 - c2) * weights))


# ── Tracker helpers ──────────────────────────────────────────────────────────

def outputs_to_bytetrack(outputs) -> np.ndarray:
    """
    Convert estimator outputs to ByteTrack input format:
    [[x1, y1, x2, y2, confidence], ...]
    """
    if not outputs:
        return np.empty((0, 6), dtype=np.float32)
    rows = []
    for p in outputs:
        x1, y1, x2, y2 = p["bbox"][:4]
        conf = float(p.get("bbox_score", p.get("score", 1.0)))
        rows.append([x1, y1, x2, y2, conf, 0])
    return np.array(rows, dtype=np.float32)


def match_output_to_track(person_output, tracks) -> int | None:
    """Return the track ID whose bbox best overlaps this person's bbox, or None."""
    if tracks is None or len(tracks) == 0:
        return None
    px1, py1, px2, py2 = person_output["bbox"][:4]
    best_iou, best_id = 0.0, None
    for t in tracks:
        tx1, ty1, tx2, ty2, tid = t[:5]
        ix1, iy1 = max(px1, tx1), max(py1, ty1)
        ix2, iy2 = min(px2, tx2), min(py2, ty2)
        inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
        if inter == 0:
            continue
        union = (px2-px1)*(py2-py1) + (tx2-tx1)*(ty2-ty1) - inter
        iou = inter / union if union > 0 else 0
        if iou > best_iou:
            best_iou, best_id = iou, int(tid)
    return best_id if best_iou > 0.3 else None


# ── Mode 2 – pick-up mid-video by jersey colour ──────────────────────────────

def process_video_track_color(
    video_path: str,
    reference_frame_idx: int,
    reference_bbox: tuple,           # (x1, y1, x2, y2) drawn around target
    output_path: str = "output_tracked_color.mp4",
    color_threshold: float = 30.0,   # lower = stricter match
):
    """
    Lets you specify a reference frame + bounding box for the target player.
    Their jersey colour is fingerprinted and then matched every frame via
    ByteTrack ID + colour distance fallback.

    Args:
        reference_frame_idx : frame number where you identify the player.
        reference_bbox      : (x1, y1, x2, y2) tight box around them.
        color_threshold     : max HSV distance to still count as a match.
    """
    tracker = ByteTrack()
    cap     = cv2.VideoCapture(video_path)
    fps     = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer  = cv2.VideoWriter(
        output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height)
    )

    ref_color  = None
    target_id  = None

    # Pre-extract reference jersey colour from the specified frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, reference_frame_idx)
    ret, ref_frame = cap.read()
    if ret:
        ref_bbox   = np.array(reference_bbox, dtype=float)
        ref_color  = get_jersey_color(ref_frame, ref_bbox)
        print(f"Reference colour (HSV): {ref_color.round(1)}")
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)   # rewind

    print(f"Mode: colour-match  |  {total} frames  |  {fps:.1f} fps")

    try:
        for frame_idx in range(total if total > 0 else int(1e9)):
            ret, frame_bgr = cap.read()
            if not ret:
                break

            outputs = estimator.process_one_image(
                cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            )
            dets   = outputs_to_bytetrack(outputs)
            tracks = tracker.update(dets, frame_bgr)

            # Try to find the target by colour each frame
            best_person, best_dist = None, float("inf")
            for p in (outputs or []):
                tid = match_output_to_track(p, tracks)
                color = get_jersey_color(frame_bgr, np.array(p["bbox"]))
                dist  = color_distance(color, ref_color) if ref_color is not None else 0

                # Prefer a known track ID, fall back to colour distance
                if tid == target_id:
                    best_person, best_dist = p, -1   # guaranteed best
                    break
                if dist < best_dist and dist < color_threshold:
                    best_person, best_dist = p, dist

            # Update target_id if we found someone
            if best_person is not None:
                tid = match_output_to_track(best_person, tracks)
                if tid is not None:
                    target_id = tid

            if best_person:
                rend = render_mesh_only([best_person], estimator.faces, height, width)
            else:
                rend = np.zeros((height, width, 3), dtype=np.uint8)

            writer.write(rend)

            if (frame_idx + 1) % 10 == 0:
                print(f"  {frame_idx + 1}/{total} frames…")
    finally:
        cap.release()
        writer.release()

    print(f"Done → {output_path}")

In [12]:
# ── Mode 1 – track from frame 0 (pick the most-central detection) ────────────

def process_video_track_first(
    video_path: str,
    output_path: str = "output_tracked_first.mp4",
):
    """
    Automatically selects the player closest to the frame centre on frame 0
    and tracks them throughout the video using ByteTrack.

    Every frame: run the full detector to get all bboxes, feed them to
    ByteTrack, extract only the target player's bbox, then pass that single
    bbox to process_one_image — skipping re-detection inside the estimator.
    """
    tracker      = ByteTrack()
    cap          = cv2.VideoCapture(video_path)
    fps          = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total        = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_centre = np.array([width / 2, height / 2])

    writer = cv2.VideoWriter(
        output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height)
    )

    target_id = None

    print(f"Mode: track-from-start  |  {total} frames  |  {fps:.1f} fps")

    try:
        for frame_idx in range(total if total > 0 else int(1e9)):
            ret, frame_bgr = cap.read()
            if not ret:
                break

            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

            # Run detector only (no 3D estimation yet) to get all bboxes
            all_boxes = estimator.detector.run_human_detection(
                frame_bgr,
                det_cat_id=0,
                bbox_thr=0.5,
                nms_thr=0.3,
                default_to_full_image=False,
            )

            if len(all_boxes) == 0:
                writer.write(np.zeros((height, width, 3), dtype=np.uint8))
                continue

            # Build fake outputs for ByteTrack (bbox + dummy confidence)
            dets = np.hstack([
                all_boxes,
                np.ones((len(all_boxes), 1), dtype=np.float32),  # conf
                np.zeros((len(all_boxes), 1), dtype=np.float32), # cls
            ])
            tracks = tracker.update(dets, frame_bgr)

            # Lock onto the most-central player on the first frame
            if target_id is None and tracks is not None and len(tracks):
                centres   = np.stack([((t[0]+t[2])/2, (t[1]+t[3])/2) for t in tracks])
                dists     = np.linalg.norm(centres - frame_centre, axis=1)
                target_id = int(tracks[np.argmin(dists), 4])
                print(f"  Locked onto track ID {target_id} at frame {frame_idx}")

            # Find the target's bbox from ByteTrack
            target_bbox = None
            if tracks is not None:
                for t in tracks:
                    if int(t[4]) == target_id:
                        target_bbox = t[:4]
                        break

            if target_bbox is None:
                writer.write(np.zeros((height, width, 3), dtype=np.uint8))
                continue

            # Pass only the target bbox to the estimator — skips internal detector
            bbox_input = np.array(target_bbox, dtype=np.float32).reshape(1, 4)
            outputs    = estimator.process_one_image(frame_rgb, bboxes=bbox_input)

            if outputs:
                rend_bgr = render_mesh_only([outputs[0]], estimator.faces, height, width)
            else:
                rend_bgr = np.zeros((height, width, 3), dtype=np.uint8)

            writer.write(rend_bgr)

            if (frame_idx + 1) % 10 == 0:
                print(f"  {frame_idx + 1}/{total} frames…")

    finally:
        cap.release()
        writer.release()

    print(f"Done → {output_path}")


### Gradio

In [19]:
!pip install --upgrade gradio

In [13]:
import gradio as gr
import tempfile
import os

# ── Helpers ──────────────────────────────────────────────────────────────────

def run_process_image(img_array):
    """Process a single uploaded image and return the rendered result."""
    if img_array is None:
        return None, "No image provided."
    outputs = estimator.process_one_image(img_array)
    if not outputs:
        return None, "No person detected."
    h, w = img_array.shape[:2]
    rend = render_mesh_only(outputs, estimator.faces, h, w)
    metrics = compute_biomechanics(outputs[0])
    lines = [
        f"L-Knee : {metrics['left_knee_deg']:.1f}°",
        f"R-Knee : {metrics['right_knee_deg']:.1f}°",
        f"L-Val  : {metrics['l_valgus']:.1f}°",
        f"R-Val  : {metrics['r_valgus']:.1f}°",
        f"KWR    : {metrics['knee_width_ratio']:.2f}",
        f"HipDrop: {metrics['hip_drop_cm']:.1f} cm",
        f"Trunk  : {metrics['trunk_lean_deg']:.1f}°",
        f"Asym   : {metrics['asym']:.1f}°",
        f"Low confidence: {metrics['low_confidence']}",
    ]
    return cv2.cvtColor(rend, cv2.COLOR_BGR2RGB), "\n".join(lines)


def run_process_video_mesh(video_path):
    if video_path is None:
        return None, "No video provided."
    out_path = os.path.join(tempfile.mkdtemp(), "output_mesh.mp4")
    process_video_mesh(video_path, out_path)
    return out_path, f"Done → {out_path}"


def run_save_obj(img_array):
    if img_array is None:
        return None, "No image provided."
    outputs = estimator.process_one_image(img_array)
    if not outputs:
        return None, "No person detected."
    out_path = os.path.join(tempfile.mkdtemp(), "mesh.obj")
    save_obj(outputs, estimator.faces, out_path)
    return out_path, f"Saved → {out_path}"


# ── Track-player helpers ──────────────────────────────────────────────────────

# Shared state between preview and tracking steps
_preview_state = {"video_path": None, "detections": [], "frame": None}


def preview_first_frame(video_path):
    """
    Extract frame 0, run the detector, draw numbered bboxes, and return
    the annotated frame so the user can identify which player to track.
    """
    if video_path is None:
        return None, "No video provided.", gr.update(maximum=0, value=0, visible=False)

    cap = cv2.VideoCapture(video_path)
    ret, frame_bgr = cap.read()
    cap.release()

    if not ret:
        return None, "Could not read video.", gr.update(visible=False)

    # Run detector on frame 0
    boxes = estimator.detector.run_human_detection(
        frame_bgr, det_cat_id=0, bbox_thr=0.5, nms_thr=0.3,
        default_to_full_image=False,
    )

    # Draw numbered bboxes
    preview = frame_bgr.copy()
    for idx, box in enumerate(boxes):
        x1, y1, x2, y2 = box[:4].astype(int)
        cv2.rectangle(preview, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            preview, str(idx),
            (x1 + 5, y1 + 30), cv2.FONT_HERSHEY_SIMPLEX,
            1.0, (0, 255, 0), 2, cv2.LINE_AA
        )

    # Store state for the tracking step
    _preview_state["video_path"] = video_path
    _preview_state["detections"] = boxes
    _preview_state["frame"]      = frame_bgr

    n = len(boxes)
    status = f"{n} player(s) detected. Enter the number of the player you want to track."
    slider = gr.update(maximum=max(n - 1, 0), value=0, visible=n > 0)

    return cv2.cvtColor(preview, cv2.COLOR_BGR2RGB), status, slider


def run_track_selected(player_idx):
    """
    Run process_video_track_first with the bbox of the selected player
    injected as the initial lock-on target.
    """
    video_path = _preview_state["video_path"]
    boxes      = _preview_state["detections"]

    if video_path is None:
        return None, "Please preview a video first."
    if len(boxes) == 0:
        return None, "No detections from preview step."

    player_idx = int(player_idx)
    if player_idx >= len(boxes):
        return None, f"Invalid player index. Choose between 0 and {len(boxes)-1}."

    selected_bbox = boxes[player_idx][:4]
    out_path = os.path.join(tempfile.mkdtemp(), "output_tracked.mp4")

    # Run tracking with the selected player's bbox as the seed
    _process_video_track_with_bbox(video_path, selected_bbox, out_path)

    return out_path, f"Done → {out_path}"


def _process_video_track_with_bbox(video_path, seed_bbox, output_path):
    """
    Same as process_video_track_first but seeds the tracker with a
    specific bbox from frame 0 instead of picking the most-central player.
    """
    from boxmot import ByteTrack

    tracker      = ByteTrack()
    cap          = cv2.VideoCapture(video_path)
    fps          = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total        = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer = cv2.VideoWriter(
        output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height)
    )

    target_id = None

    try:
        for frame_idx in range(total if total > 0 else int(1e9)):
            ret, frame_bgr = cap.read()
            if not ret:
                break

            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

            # Run detector to get all current bboxes
            all_boxes = estimator.detector.run_human_detection(
                frame_bgr, det_cat_id=0, bbox_thr=0.5, nms_thr=0.3,
                default_to_full_image=False,
            )

            if len(all_boxes) == 0:
                writer.write(np.zeros((height, width, 3), dtype=np.uint8))
                continue

            dets = np.hstack([
                all_boxes,
                np.ones((len(all_boxes), 1), dtype=np.float32),
                np.zeros((len(all_boxes), 1), dtype=np.float32),
            ])
            tracks = tracker.update(dets, frame_bgr)

            # On frame 0, lock onto the track whose bbox best overlaps seed_bbox
            if target_id is None and tracks is not None and len(tracks):
                if frame_idx == 0:
                    # Match seed_bbox via IoU
                    best_iou, best_id = 0.0, None
                    sx1, sy1, sx2, sy2 = seed_bbox
                    for t in tracks:
                        tx1, ty1, tx2, ty2, tid = t[:5]
                        ix1, iy1 = max(sx1, tx1), max(sy1, ty1)
                        ix2, iy2 = min(sx2, tx2), min(sy2, ty2)
                        inter = max(0, ix2-ix1) * max(0, iy2-iy1)
                        if inter == 0: continue
                        union = (sx2-sx1)*(sy2-sy1) + (tx2-tx1)*(ty2-ty1) - inter
                        iou = inter / (union + 1e-8)
                        if iou > best_iou:
                            best_iou, best_id = iou, int(tid)
                    target_id = best_id
                    print(f"  Locked onto track ID {target_id} (IoU={best_iou:.2f})")

            # Find target bbox from ByteTrack
            target_bbox = None
            if tracks is not None:
                for t in tracks:
                    if int(t[4]) == target_id:
                        target_bbox = t[:4]
                        break

            if target_bbox is None:
                writer.write(np.zeros((height, width, 3), dtype=np.uint8))
                continue

            bbox_input = np.array(target_bbox, dtype=np.float32).reshape(1, 4)
            outputs    = estimator.process_one_image(frame_rgb, bboxes=bbox_input)

            if outputs:
                rend_bgr = render_mesh_only([outputs[0]], estimator.faces, height, width)
            else:
                rend_bgr = np.zeros((height, width, 3), dtype=np.uint8)

            writer.write(rend_bgr)

            if (frame_idx + 1) % 10 == 0:
                print(f"  {frame_idx + 1}/{total} frames…")

    finally:
        cap.release()
        writer.release()


# ── UI ────────────────────────────────────────────────────────────────────────

with gr.Blocks(title="SAM 3D Body — Soccer Analysis") as demo:

    gr.Markdown("# SAM 3D Body — Soccer Biomechanics Prototype")

    # ── Tab 1: Single image ──────────────────────────────────────────────────
    with gr.Tab("Single Image"):
        gr.Markdown("Upload an image to reconstruct the 3D body mesh and compute biomechanical metrics.")
        with gr.Row():
            with gr.Column():
                img_input   = gr.Image(label="Input Image", type="numpy")
                img_btn     = gr.Button("Run", variant="primary")
            with gr.Column():
                img_output  = gr.Image(label="Reconstruction")
                img_metrics = gr.Textbox(label="Metrics", lines=10)
        img_btn.click(run_process_image, inputs=img_input, outputs=[img_output, img_metrics])

    # ── Tab 2: Video — all players ───────────────────────────────────────────
    with gr.Tab("Video — All Players"):
        gr.Markdown("Upload a video to render the 3D mesh for all detected players.")
        with gr.Row():
            with gr.Column():
                vid_mesh_input  = gr.Video(label="Input Video")
                vid_mesh_btn    = gr.Button("Run", variant="primary")
            with gr.Column():
                vid_mesh_output = gr.Video(label="Output Video")
                vid_mesh_status = gr.Textbox(label="Status")
        vid_mesh_btn.click(
            run_process_video_mesh,
            inputs=vid_mesh_input,
            outputs=[vid_mesh_output, vid_mesh_status]
        )

    # ── Tab 3: Video — track selected player ─────────────────────────────────
    with gr.Tab("Video — Track Player"):
        gr.Markdown(
            "**Step 1** — Upload a video and click **Preview** to detect players on frame 0.  \n"
            "**Step 2** — Enter the number shown on the player you want to track.  \n"
            "**Step 3** — Click **Track** to process the full video."
        )
        with gr.Row():
            with gr.Column():
                track_video_input = gr.Video(label="Input Video")
                preview_btn       = gr.Button("Step 1 — Preview", variant="secondary")
                track_preview_out = gr.Image(label="Frame 0 — pick a player number")
                preview_status    = gr.Textbox(label="Status")
                player_slider     = gr.Slider(
                    minimum=0, maximum=0, step=1, value=0,
                    label="Player index", visible=False
                )
                track_btn = gr.Button("Step 2 — Track", variant="primary")
            with gr.Column():
                track_video_output = gr.Video(label="Output Video")
                track_status       = gr.Textbox(label="Status")

        preview_btn.click(
            preview_first_frame,
            inputs=track_video_input,
            outputs=[track_preview_out, preview_status, player_slider]
        )
        track_btn.click(
            run_track_selected,
            inputs=player_slider,
            outputs=[track_video_output, track_status]
        )

    # ── Tab 4: Export .obj ───────────────────────────────────────────────────
    with gr.Tab("Export .obj"):
        gr.Markdown("Upload an image to export the reconstructed 3D mesh as a `.obj` file.")
        with gr.Row():
            with gr.Column():
                obj_input  = gr.Image(label="Input Image", type="numpy")
                obj_btn    = gr.Button("Export", variant="primary")
            with gr.Column():
                obj_output = gr.File(label="Download .obj")
                obj_status = gr.Textbox(label="Status")
        obj_btn.click(run_save_obj, inputs=obj_input, outputs=[obj_output, obj_status])

In [18]:
demo.launch(inline=True, share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [19]:
demo.close()

Closing server running on port: 7860


### IPyWidgets

In [31]:
import os
import io
import cv2
import tempfile
import numpy as np
import ipywidgets as widgets
from PIL import Image as PILImage
from IPython.display import display, clear_output

# ------------------------------------------------------------
# CONFIG: set your video filename here (same folder as notebook)
# ------------------------------------------------------------
DEFAULT_VIDEO = "test_data/match.mp4"   # <-- change this to your real filename

# ------------------------------------------------------------
# Helper: numpy RGB image -> widget image
# ------------------------------------------------------------
def np_to_widget(img_rgb: np.ndarray, width=720):
    pil = PILImage.fromarray(img_rgb.astype(np.uint8))
    buf = io.BytesIO()
    pil.save(buf, format="PNG")
    return widgets.Image(value=buf.getvalue(), format="png", width=width)

# ------------------------------------------------------------
# Shared state
# ------------------------------------------------------------
_track_state = {
    "video_path": None,
    "detections": None,
}

# ------------------------------------------------------------
# UI
# ------------------------------------------------------------
video_name = widgets.Text(
    value=DEFAULT_VIDEO,
    description="Video:",
    layout=widgets.Layout(width="500px")
)

preview_btn = widgets.Button(
    description="Preview Players",
    button_style="warning"
)

player_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=0,
    step=1,
    description="Player:",
    disabled=True,
    layout=widgets.Layout(width="600px")
)

track_btn = widgets.Button(
    description="Track Selected Player",
    button_style="primary",
    disabled=True
)

out_preview = widgets.Output()
out_result = widgets.Output()
out_status = widgets.Output()

def show_status(msg):
    with out_status:
        clear_output(wait=True)
        print(msg)

# ------------------------------------------------------------
# Step 1: Read first frame + detect players
# ------------------------------------------------------------
def on_preview(_):
    filename = video_name.value.strip()

    if not filename:
        show_status("Please enter a video filename.")
        return

    video_path = os.path.abspath(filename)

    if not os.path.exists(video_path):
        show_status(f"Video not found: {video_path}")
        return

    show_status("Opening video file...")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        show_status(f"OpenCV could not open video:\n{video_path}")
        return

    show_status("Reading first frame...")

    ret, frame_bgr = cap.read()
    cap.release()

    if not ret or frame_bgr is None:
        show_status("Could not read the first frame.")
        return

    show_status(f"First frame loaded: shape={frame_bgr.shape}")

    # OPTIONAL: show raw first frame immediately so we know video loading works
    with out_preview:
        clear_output(wait=True)
        display(np_to_widget(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)))

    show_status("Running person detection on first frame...")

    try:
        boxes = estimator.detector.run_human_detection(
            frame_bgr,
            det_cat_id=0,
            bbox_thr=0.5,
            nms_thr=0.3,
            default_to_full_image=False,
        )
    except Exception as e:
        show_status(f"Detection crashed:\n{type(e).__name__}: {e}")
        return

    show_status(f"Detection finished. Found {len(boxes)} player(s).")

    preview = frame_bgr.copy()

    for idx, box in enumerate(boxes):
        x1, y1, x2, y2 = box[:4].astype(int)
        cv2.rectangle(preview, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            preview, str(idx),
            (x1 + 5, y1 + 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.0,
            (0, 255, 0),
            2,
            cv2.LINE_AA
        )

    _track_state["video_path"] = video_path
    _track_state["detections"] = boxes

    n = len(boxes)
    player_slider.max = max(n - 1, 0)
    player_slider.value = 0
    player_slider.disabled = (n == 0)
    track_btn.disabled = (n == 0)

    with out_preview:
        clear_output(wait=True)
        display(np_to_widget(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB)))

    if n == 0:
        show_status("No players detected in first frame.")
    else:
        show_status(f"Detected {n} player(s). Select one and click Track.")

# ------------------------------------------------------------
# Step 2: Track selected player
# ------------------------------------------------------------
def on_track(_):
    video_path = _track_state["video_path"]
    boxes = _track_state["detections"]

    if video_path is None or boxes is None or len(boxes) == 0:
        show_status("Please run Preview Players first.")
        return

    selected_idx = player_slider.value
    selected_bbox = boxes[selected_idx][:4]

    base, ext = os.path.splitext(video_path)
    out_path = f"{base}_tracked{ext}"

    show_status(f"Tracking player {selected_idx}... this may take a while.")

    # Uses your existing notebook function
    _process_video_track_with_bbox(video_path, selected_bbox, out_path)

    with out_result:
        clear_output(wait=True)
        print("Tracking complete.")
        print(f"Saved video: {out_path}")

    show_status("Done.")

preview_btn.on_click(on_preview)
track_btn.on_click(on_track)

display(widgets.VBox([
    widgets.HTML("<h3>Track a Player from Local Video</h3>"),
    video_name,
    preview_btn,
    out_preview,
    player_slider,
    track_btn,
    out_result,
    out_status
]))

### Tests

In [ ]:
rend_img = process_image("/kaggle/input/datasets/aminefezzani/sam-3d-body/a.JPG")

In [ ]:
rend_img.shape, rend_img.dtype
import matplotlib.pyplot as plt

plt.figure(figsize=(20,20))
plt.imshow(rend_img[:, :, ::-1] / 255)
plt.axis('off')
plt.show()

In [ ]:
save_obj(outputs, estimator.faces, "frame_0042.obj")

In [ ]:
# ── Example usage ────────────────────────────────────────────────────────────

VIDEO = "/kaggle/input/datasets/aminefezzani/sam-3d-body/8.mp4"

"""
Choose if you want to render everything, render just the mesh for multiple persons, or render with tracking the center of the frame

process_video("/kaggle/input/datasets/aminefezzani/sam-3d-body/8.mp4")
process_video_mesh("/kaggle/input/datasets/aminefezzani/sam-3d-body/8.mp4", "output_mesh.mp4")
"""

# Mode 1: automatically lock onto the most-central player from frame 0
process_video_track_first(VIDEO, "output_track_first.mp4")

# Mode 2: fingerprint by jersey colour — specify which frame and bbox
# process_video_track_color(
#     VIDEO,
#     reference_frame_idx = 120,          # frame where you can clearly see them
#     reference_bbox      = (410, 80, 560, 340),  # (x1, y1, x2, y2)
#     output_path         = "output_track_color.mp4",
#     color_threshold     = 30.0,
# )